In [ ]:
import sys
!{sys.executable} -m pip install DECIMER
from DECIMER import predict_SMILES

smiles = predict_SMILES(r"/content/molecule.png")

print(smiles)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.8/48.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.7/5.7 MB 54.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.8/61.8 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 495.6/495.6 kB 33.8 MB/s eta 0:00:00


/root/.data/DECIMER-V2/models.zip
... done downloading trained model!


/root/.data/DECIMER-V2/DECIMER_HandDrawn_model.zip
... done downloading trained model!
CN1C=NC2=C1C(=O)N(C)C(=O)N2C


In [ ]:
import os
import cv2
import numpy as np
from PIL import Image

!pip install rdkit

# ---------------------------------------------------------------- cleaning
def _strip_frame(gray, dark=70, area_frac=0.03):
    """Set large dark blobs touching the edge (screenshot frame/border) to white."""
    mask = (gray < dark).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(mask, 8)
    H, W = gray.shape
    out = gray.copy()
    for i in range(1, n):
        x, y, w, h, a = st[i]
        if (x == 0 or y == 0 or x + w == W or y + h == H) and a > area_frac * H * W:
            out[lab == i] = 255
    return out


def clean_for_decimer(src, binarize=True):
    """Background-flatten + binarise a photographed structure so DECIMER can read it."""
    img = cv2.imread(src) if isinstance(src, str) else src
    # Check if image was loaded successfully
    if img is None:
        raise FileNotFoundError(f"Could not read image: {src}. Please ensure the path is correct and the file exists.")

    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    H, W = gray.shape
    is_clean = gray.mean() > 235 and gray.std() < 45

    if not is_clean:
        gray = _strip_frame(gray)
        if max(H, W) < 500:
            pil = Image.fromarray(gray)
            gray = np.array(pil.resize((pil.width * 2, pil.height * 2), Image.LANCZOS))
        k = max(15, (min(gray.shape) // 12) | 1)
        bg = cv2.morphologyEx(gray, cv2.MORPH_CLOSE,
                              cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k)))
        gray = cv2.normalize(cv2.divide(gray, bg, scale=255), None, 0, 255,
                             cv2.NORM_MINMAX).astype(np.uint8)
        gray = cv2.bilateralFilter(gray, 7, 50, 50)

    if not binarize:
        return gray
    _, b = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return b


# ---------------------------------------------------------------- recognise
def read_smiles(image_path):
    """Run DECIMER on a cleaned image. Returns SMILES or None if DECIMER is unavailable."""
    try:
        from DECIMER import predict_SMILES
    except Exception:
        print("DECIMER not installed here; pass the SMILES to redraw() yourself.")
        return None
    return predict_SMILES(image_path)


# ---------------------------------------------------------------- redraw
def redraw(smiles, out_base, width=1600, height=1200, bond_width=4, svg=True):
    """Render a SMILES as a crisp high-resolution PNG (and SVG). This is the sharp image."""
    from rdkit import Chem
    from rdkit.Chem.Draw import rdMolDraw2D
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"RDKit could not parse SMILES: {smiles}")

    d = rdMolDraw2D.MolDraw2DCairo(width, height)
    d.drawOptions().bondLineWidth = bond_width
    rdMolDraw2D.PrepareAndDrawMolecule(d, mol)
    d.FinishDrawing()
    png = out_base + "_highres.png"
    with open(png, "wb") as f:
        f.write(d.GetDrawingText())

    if svg:
        d2 = rdMolDraw2D.MolDraw2DSVG(width, height)
        d2.drawOptions().bondLineWidth = bond_width
        rdMolDraw2D.PrepareAndDrawMolecule(d2, mol)
        d2.FinishDrawing()
        with open(out_base + "_highres.svg", "w") as f:
            f.write(d2.GetDrawingText())
    return png


# ---------------------------------------------------------------- full pipeline
def photo_to_highres(image_path, out_dir="highres_out", width=1600, height=1200):
    """photo -> clean -> DECIMER -> high-res redraw. Returns (smiles, png_path)."""
    os.makedirs(out_dir, exist_ok=True)
    base = os.path.splitext(os.path.basename(image_path))[0]

    cleaned = clean_for_decimer(image_path)
    cleaned_path = os.path.join(out_dir, base + "_cleaned.png")
    cv2.imwrite(cleaned_path, cleaned)             # this is what DECIMER reads

    smiles = read_smiles(cleaned_path)
    if not smiles:
        print("No SMILES (DECIMER missing). Cleaned image saved at:", cleaned_path)
        return None, cleaned_path

    png = redraw(smiles, os.path.join(out_dir, base), width, height)
    print(f"SMILES: {smiles}\nHigh-res image: {png}")
    return smiles, png

print("\n--- Running photo_to_highres on molecule, molecule1 ... molecule5 ---")

IMAGE_DIR = "/content"
OUT_DIR   = "/content/highres_output"
NAMES     = ["molecule", "molecule1", "molecule2",
             "molecule3", "molecule4", "molecule5"]   # molecule3 skipped if absent

for name in NAMES:
    path = None
    for ext in (".png", ".jpg", ".jpeg"):
        cand = os.path.join(IMAGE_DIR, name + ext)
        if os.path.exists(cand):
            path = cand
            break
    if path is None:
        print(f"[skip] {name}: not found")
        continue

    print(f"\n[run] {name}")
    smiles_result, highres_path = photo_to_highres(path, out_dir=OUT_DIR)
    if smiles_result:
        print(f"  Done. SMILES: {smiles_result}  ->  {highres_path}")
    else:
        print(f"  Cleaned only (DECIMER not available); cleaned image at: {highres_path}")


--- Running photo_to_highres on molecule, molecule1 ... molecule5 ---

[run] molecule
SMILES: CN1C=NC2=C1C(=O)N(C)C(=O)N2C
High-res image: /content/highres_output/molecule_highres.png
  Done. SMILES: CN1C=NC2=C1C(=O)N(C)C(=O)N2C  ->  /content/highres_output/molecule_highres.png

[run] molecule1
SMILES: CN1C=NC2=C1C(=O)N(C)C(=O)N2C.[I-].[I-]
High-res image: /content/highres_output/molecule1_highres.png
  Done. SMILES: CN1C=NC2=C1C(=O)N(C)C(=O)N2C.[I-].[I-]  ->  /content/highres_output/molecule1_highres.png

[run] molecule2
SMILES: C1=CC=CC=C1.[GaH3].[GaH3].[I-].[I-].[I-].[I-].[I-]
High-res image: /content/highres_output/molecule2_highres.png
  Done. SMILES: C1=CC=CC=C1.[GaH3].[GaH3].[I-].[I-].[I-].[I-].[I-]  ->  /content/highres_output/molecule2_highres.png
[skip] molecule3: not found

[run] molecule4
SMILES: CC(=O)OCC1=C(C=CC=C1)C(=O)O.[I-]
High-res image: /content/highres_output/molecule4_highres.png
  Done. SMILES: CC(=O)OCC1=C(C=CC=C1)C(=O)O.[I-]  ->  /content/highres_output/molecu

In [ ]:
import sys
!{sys.executable} -m pip install DECIMER rdkit

from DECIMER import predict_SMILES
import cv2, numpy as np
from PIL import Image
from rdkit import Chem

# ---- clean the image before DECIMER reads it ----
def clean(src, binarize=True):
    img = cv2.imread(src)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    H, W = gray.shape
    is_clean = gray.mean() > 235 and gray.std() < 45        # already a clean drawing?
    if not is_clean:
        if max(H, W) < 500:                                  # upscale tiny images
            pil = Image.fromarray(gray)
            gray = np.array(pil.resize((pil.width*2, pil.height*2), Image.LANCZOS))
        k = max(15, (min(gray.shape)//12) | 1)               # flat-field: remove paper/stains
        bg = cv2.morphologyEx(gray, cv2.MORPH_CLOSE,
                              cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k)))
        gray = cv2.normalize(cv2.divide(gray, bg, scale=255), None, 0, 255,
                             cv2.NORM_MINMAX).astype(np.uint8)
        gray = cv2.bilateralFilter(gray, 7, 50, 50)
    if not binarize:
        return gray
    _, b = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return b

# ---- run it ----
raw = "/content/molecule.png"
cv2.imwrite("/content/molecule_clean.png", clean(raw))       # <-- the change: clean first

smiles = predict_SMILES("/content/molecule_clean.png")       # feed the CLEANED image
print("SMILES:", smiles)

# ---- quick trust check ----
m = Chem.MolFromSmiles(smiles)
print("valid:", m is not None)
print("canonical:", Chem.MolToSmiles(m) if m else "INVALID - DECIMER likely misread")

SMILES: CN1C=NC2=C1C(=O)N(C)C(=O)N2C
valid: True
canonical: Cn1c(=O)c2c(ncn2C)n(C)c1=O


In [ ]:
import os
import cv2
import numpy as np
from PIL import Image

# ---------------------------------------------------------------- internal: clean
def _strip_frame(gray, dark=70, area_frac=0.03):
    """Set large dark blobs touching the edge (screenshot frame/border) to white."""
    mask = (gray < dark).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(mask, 8)
    H, W = gray.shape
    out = gray.copy()
    for i in range(1, n):
        x, y, w, h, a = st[i]
        if (x == 0 or y == 0 or x + w == W or y + h == H) and a > area_frac * H * W:
            out[lab == i] = 255
    return out


def _clean(src, binarize=True):
    """Background-flatten + binarise so DECIMER can read the structure. Runs internally."""
    img = cv2.imread(src) if isinstance(src, str) else src
    if img is None:
        raise FileNotFoundError(f"Could not read image: {src}")
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY) if img.ndim == 3 else img
    H, W = gray.shape
    is_clean = gray.mean() > 235 and gray.std() < 45      # already a clean digital drawing?

    if not is_clean:
        gray = _strip_frame(gray)
        if max(H, W) < 500:                                # upscale tiny images
            pil = Image.fromarray(gray)
            gray = np.array(pil.resize((pil.width * 2, pil.height * 2), Image.LANCZOS))
        k = max(15, (min(gray.shape) // 12) | 1)           # flat-field: remove paper/stains/gradient
        bg = cv2.morphologyEx(gray, cv2.MORPH_CLOSE,
                              cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k)))
        gray = cv2.normalize(cv2.divide(gray, bg, scale=255), None, 0, 255,
                             cv2.NORM_MINMAX).astype(np.uint8)
        gray = cv2.bilateralFilter(gray, 7, 50, 50)

    if not binarize:
        return gray
    _, b = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    return b


# ---------------------------------------------------------------- internal: redraw
def _redraw(smiles, out_base, width=1600, height=1200, bond_width=4, svg=True):
    """Render a SMILES as a crisp high-resolution PNG (and SVG). This is the sharp image."""
    from rdkit import Chem
    from rdkit.Chem.Draw import rdMolDraw2D
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        raise ValueError(f"RDKit could not parse SMILES: {smiles}")

    d = rdMolDraw2D.MolDraw2DCairo(width, height)
    d.drawOptions().bondLineWidth = bond_width
    rdMolDraw2D.PrepareAndDrawMolecule(d, mol)
    d.FinishDrawing()
    png = out_base + "_highres.png"
    with open(png, "wb") as f:
        f.write(d.GetDrawingText())

    if svg:
        d2 = rdMolDraw2D.MolDraw2DSVG(width, height)
        d2.drawOptions().bondLineWidth = bond_width
        rdMolDraw2D.PrepareAndDrawMolecule(d2, mol)
        d2.FinishDrawing()
        with open(out_base + "_highres.svg", "w") as f:
            f.write(d2.GetDrawingText())
    return png


# ================================================================
# THE ONE FUNCTION: raw image in  ->  high-res image out
# (cleaning + DECIMER happen inside; you never call clean separately)
# ================================================================
def raw_to_highres(image_path, out_dir="/content/highres_output",
                   width=1600, height=1200, keep_cleaned=False):
    from DECIMER import predict_SMILES
    os.makedirs(out_dir, exist_ok=True)
    base = os.path.splitext(os.path.basename(image_path))[0]

    # 1) clean internally (DECIMER reads from a file path, so write a temp cleaned image)
    cleaned = _clean(image_path)
    cleaned_path = os.path.join(out_dir, base + "_cleaned.png")
    cv2.imwrite(cleaned_path, cleaned)

    # 2) recognise
    smiles = predict_SMILES(cleaned_path)

    # 3) redraw crisp high-res
    png = _redraw(smiles, os.path.join(out_dir, base), width, height)

    if not keep_cleaned and os.path.exists(cleaned_path):
        os.remove(cleaned_path)      # discard the intermediate; set keep_cleaned=True to keep it

    print(f"{base}: {smiles}  ->  {png}")
    return smiles, png


# ---------------------------------------------------------------- run on all images
IMAGE_DIR = "/content"
OUT_DIR   = "/content/highres_output"
NAMES = ["molecule", "molecule1", "molecule2", "molecule3", "molecule4", "molecule5","blurry1_aspirin","mix1_aspirin_lowers","mix2_paracetamol_blur","mix3_ibuprofen_paper","mix4_caffeine_scan"]

for name in NAMES:
    path = None
    for ext in (".png", ".jpg", ".jpeg"):
        cand = os.path.join(IMAGE_DIR, name + ext)
        if os.path.exists(cand):
            path = cand
            break
    if path is None:
        print(f"[skip] {name}: not found")
        continue
    try:
        raw_to_highres(path, out_dir=OUT_DIR)
    except Exception as e:
        print(f"[error] {name}: {e}")

molecule: CN1C=NC2=C1C(=O)N(C)C(=O)N2C  ->  /content/highres_output/molecule_highres.png
molecule1: CN1C=NC2=C1C(=O)N(C)C(=O)N2C.[I-].[I-]  ->  /content/highres_output/molecule1_highres.png
molecule2: C1=CC=CC=C1.[GaH3].[GaH3].[I-].[I-].[I-].[I-].[I-]  ->  /content/highres_output/molecule2_highres.png
[skip] molecule3: not found
molecule4: CC(=O)OCC1=C(C=CC=C1)C(=O)O.[I-]  ->  /content/highres_output/molecule4_highres.png
molecule5: C(C1[C@H]([C@@H](C(C(O)O1)O)O)O)O  ->  /content/highres_output/molecule5_highres.png
blurry1_aspirin: CC(C)CC1CCCCC1C(C)C  ->  /content/highres_output/blurry1_aspirin_highres.png
[skip] mix1_aspirin_lowers: not found
mix2_paracetamol_blur: C=C(C)C(=C)C1=CC=C(C=C1)C#N  ->  /content/highres_output/mix2_paracetamol_blur_highres.png
mix3_ibuprofen_paper: CC(C)CC1=CC=C(C=C1)C(C)C(=O)O  ->  /content/highres_output/mix3_ibuprofen_paper_highres.png
mix4_caffeine_scan: CN1C=NC2=C1C(=O)N(C)C(=O)N2C  ->  /content/highres_output/mix4_caffeine_scan_highres.png


In [ ]:
import os
import pandas as pd
from DECIMER import predict_SMILES

image_folder = r"/content/"
output_csv = r"/content/results.csv"

results = []

# Get a list of all image files in the specified folder
# Consider common image extensions
image_files = [f for f in os.listdir(image_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff'))]

for filename in image_files:
    image_path = os.path.join(image_folder, filename)
    try:
        smiles = predict_SMILES(image_path)
        results.append({'Image_Name': filename, 'SMILES': smiles})
    except Exception as e:
        results.append({'Image_Name': filename, 'SMILES': f"Error: {e}"})

# Create a DataFrame and save to CSV
df = pd.DataFrame(results)
df.to_csv(output_csv, index=False)
print(f"Results saved to {output_csv}")

Results saved to /content/results.csv


In [ ]:
from rdkit.Chem.Draw import rdMolDraw2D

IMAGE_DIR   = "/content"                 # folder with your images
OUT_DIR     = "/content/output"          # results go here
TRY_HAND_DRAWN = True                    # also try DECIMER's hand-drawn model (slower, helps sketches)
USE_PUBCHEM    = True                    # verify reads against PubChem (needs internet)

# OPTIONAL: known correct SMILES to measure real accuracy. Leave {} if you don't have them.
GROUND_TRUTH = {
    # "molecule": "CN1C=NC2=C1C(=O)N(C)C(=O)N2C",
    # "mix1_aspirin_lowres": "CC(=O)Oc1ccccc1C(=O)O",
}
os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
def _strip_frame(gray, dark=70, area_frac=0.03):
    mask=(gray<dark).astype(np.uint8); n,lab,st,_=cv2.connectedComponentsWithStats(mask,8)
    H,W=gray.shape; out=gray.copy()
    for i in range(1,n):
        x,y,w,h,a=st[i]
        if (x==0 or y==0 or x+w==W or y+h==H) and a>area_frac*H*W: out[lab==i]=255
    return out

def clean_image(src, binarize=True):
    img=cv2.imread(src) if isinstance(src,str) else src
    if img is None: raise FileNotFoundError(src)
    gray=cv2.cvtColor(img,cv2.COLOR_BGR2GRAY) if img.ndim==3 else img
    H,W=gray.shape
    is_clean = gray.mean()>235 and gray.std()<45          # already a clean digital drawing?
    if not is_clean:
        gray=_strip_frame(gray)
        if max(H,W)<500:                                  # upscale tiny images
            pil=Image.fromarray(gray); gray=np.array(pil.resize((pil.width*2,pil.height*2),Image.LANCZOS))
        k=max(15,(min(gray.shape)//12)|1)                 # flat-field: remove paper/stains/gradient
        bg=cv2.morphologyEx(gray,cv2.MORPH_CLOSE,cv2.getStructuringElement(cv2.MORPH_ELLIPSE,(k,k)))
        gray=cv2.normalize(cv2.divide(gray,bg,scale=255),None,0,255,cv2.NORM_MINMAX).astype(np.uint8)
        gray=cv2.bilateralFilter(gray,7,50,50)
    if not binarize: return gray
    _,b=cv2.threshold(gray,0,255,cv2.THRESH_BINARY+cv2.THRESH_OTSU); return b

In [ ]:
# 4) Recognise: multi-variant CONSENSUS + salt-stripping (the accuracy engine)
from collections import Counter
from rdkit.Chem.MolStandardize import rdMolStandardize
_chooser = rdMolStandardize.LargestFragmentChooser()
IMPROBABLE = set("Ga Ge As Se Sb Te Sn Pb Hg Tl Bi Al Si B Zn Fe Cu Ni Co Mn Cr".split())
_pubchem_cache = {}

def _variants(image_path, work_dir):
    os.makedirs(work_dir, exist_ok=True)
    base = os.path.splitext(os.path.basename(image_path))[0]
    v = {"original": image_path}
    bp = os.path.join(work_dir, base + "_bin.png");  cv2.imwrite(bp, clean_image(image_path, True))
    gp = os.path.join(work_dir, base + "_gray.png"); cv2.imwrite(gp, clean_image(image_path, False))
    v["cleaned_binary"] = bp; v["cleaned_gray"] = gp
    return v

def _decimer(path, hand_drawn=False):
    from DECIMER import predict_SMILES
    for kw in ({"confidence": True, "hand_drawn": hand_drawn}, {"confidence": True},
               {"hand_drawn": hand_drawn}, {}):
        try:
            out = predict_SMILES(path, **kw)
            if isinstance(out, tuple):
                try: conf = float(np.mean([c[1] for c in out[1]]))
                except Exception: conf = None
                return out[0], conf
            return out, None
        except TypeError:
            continue
    return None, None

def _sanitize(smiles):
    if not smiles: return None, {}
    m = Chem.MolFromSmiles(smiles)
    if m is None: return None, {}
    n_frag = len(Chem.GetMolFrags(m))
    m = _chooser.choose(m)
    elems = {a.GetSymbol() for a in m.GetAtoms()}
    return Chem.MolToSmiles(m), {
        "improbable": sorted(elems & IMPROBABLE),
        "halogens": sorted(elems & set("F Cl Br I".split())),
        "had_fragments": n_frag > 1,
    }

def _pubchem_ok(smiles):
    if not USE_PUBCHEM or not smiles: return None
    if smiles in _pubchem_cache: return _pubchem_cache[smiles]
    import time
    try:
        import pubchempy as pcp
        from rdkit.Chem import inchi as rdi
        ik = rdi.InchiToInchiKey(rdi.MolToInchi(Chem.MolFromSmiles(smiles)))
        ans = None
        for _ in range(3):
            try:
                ans = bool(pcp.get_compounds(ik, "inchikey")); break
            except Exception:
                time.sleep(1.0)
        _pubchem_cache[smiles] = ans
        return ans
    except Exception:
        return None

def recognize(image_path, work_dir):
    reads = []
    for vpath in _variants(image_path, work_dir).values():
        for hd in ([False, True] if TRY_HAND_DRAWN else [False]):
            smi, conf = _decimer(vpath, hand_drawn=hd)
            clean, info = _sanitize(smi)
            if clean and not info.get("improbable"):
                reads.append((clean, conf or 0.0, bool(info.get("halogens")),
                              bool(info.get("had_fragments"))))
    if not reads:
        return {"smiles": None, "trust": False, "votes": 0, "total": 0,
                "agreement": 0, "confidence": None, "needs_review": True}

    votes = Counter(r[0] for r in reads)
    best, n = votes.most_common(1)[0]
    total = len(reads)
    best_conf = max(r[1] for r in reads if r[0] == best)
    had_halogen = any(r[2] for r in reads if r[0] == best)
    agreement = n / total

    in_pc = _pubchem_ok(best)
    trust = (n >= 2 and agreement >= 0.5) or (best_conf >= 0.9 and len(votes) == 1)
    if in_pc is False and n < 2:
        trust = False
    needs_review = (not trust) or had_halogen

    return {"smiles": best, "trust": bool(trust), "votes": n, "total": total,
            "agreement": round(agreement, 2), "confidence": round(best_conf, 3),
            "in_pubchem": in_pc, "needs_review": bool(needs_review)}

In [ ]:
def redraw(smiles, out_base, width=1600, height=1200, bond_width=4):
    mol=Chem.MolFromSmiles(smiles)
    if mol is None: return None
    d=rdMolDraw2D.MolDraw2DCairo(width,height); d.drawOptions().bondLineWidth=bond_width
    rdMolDraw2D.PrepareAndDrawMolecule(d,mol); d.FinishDrawing()
    png=out_base+"_highres.png"; open(png,"wb").write(d.GetDrawingText())
    d2=rdMolDraw2D.MolDraw2DSVG(width,height); d2.drawOptions().bondLineWidth=bond_width
    rdMolDraw2D.PrepareAndDrawMolecule(d2,mol); d2.FinishDrawing()
    open(out_base+"_highres.svg","w").write(d2.GetDrawingText())
    return png

In [ ]:
# 6) Run on every image -> CSV + accuracy summary  (always redraws; trust is a LABEL)
work_dir = os.path.join(OUT_DIR, "_tmp")
rows = []; n = correct = trusted = 0

image_files = sorted(f for f in os.listdir(IMAGE_DIR)
                     if f.lower().endswith((".png", ".jpg", ".jpeg", ".bmp", ".tiff"))
                     and "_highres" not in f and "_cleaned" not in f)

for fname in image_files:
    path = os.path.join(IMAGE_DIR, fname); name = os.path.splitext(fname)[0]; n += 1
    try:
        best = recognize(path, work_dir)
    except Exception as e:
        rows.append({"image": fname, "smiles": f"ERROR: {e}", "trust": False,
                     "needs_review": True}); continue

    # ALWAYS draw the best structure so you can see every result; trust just tells you whether to double-check
    highres = redraw(best["smiles"], os.path.join(OUT_DIR, name)) if best["smiles"] else None

    correct_flag = None
    if name in GROUND_TRUTH:
        gt = Chem.MolFromSmiles(GROUND_TRUTH[name])
        gt = Chem.MolToSmiles(gt) if gt else None
        correct_flag = int(best["smiles"] == gt); correct += correct_flag
    trusted += int(best["trust"])

    rows.append({"image": fname, "smiles": best["smiles"], "votes": f"{best['votes']}/{best['total']}",
                 "agreement": best["agreement"], "confidence": best["confidence"],
                 "in_pubchem": best.get("in_pubchem"), "trust": best["trust"],
                 "needs_review": best["needs_review"], "highres": highres, "correct": correct_flag})
    tag = "OK   " if best["trust"] else "CHECK"
    print(f"[{tag}] {fname:28s} {best['votes']}/{best['total']} agree  {best['smiles']}")

df = pd.DataFrame(rows); df.to_csv(os.path.join(OUT_DIR, "results.csv"), index=False)
print("\n================ SUMMARY ================")
print(f"images processed : {n}")
print(f"trusted          : {trusted}/{n}  (others are produced too, just flagged needs_review)")
n_gt = sum(1 for r in rows if r.get('correct') is not None)
if n_gt:
    print(f"EXACT-MATCH ACC  : {correct}/{n_gt} = {100*correct/n_gt:.0f}%   (vs GROUND_TRUTH)")
else:
    print("EXACT-MATCH ACC  : fill GROUND_TRUTH in cell 2 to measure real accuracy")
print(f"\nResults CSV: {os.path.join(OUT_DIR, 'results.csv')}")
df

[13:02:00] Running LargestFragmentChooser
[13:02:04] Running LargestFragmentChooser
[13:02:08] Running LargestFragmentChooser
[13:02:11] Running LargestFragmentChooser
[13:02:14] Running LargestFragmentChooser
[13:02:18] Running LargestFragmentChooser


[OK   ] blurry1_aspirin.png          3/6 agree  CC(C)C[C@@H]1CCCCC1C(C)C


[13:02:22] Running LargestFragmentChooser
[13:02:26] Running LargestFragmentChooser
[13:02:29] Running LargestFragmentChooser
[13:02:34] Running LargestFragmentChooser
[13:02:37] Running LargestFragmentChooser
[13:02:40] Running LargestFragmentChooser


[OK   ] mix1_aspirin_lowres.png      5/6 agree  CC(=O)Oc1ccccc1C(=O)O


[13:02:43] Running LargestFragmentChooser
[13:02:49] Running LargestFragmentChooser
[13:02:53] Running LargestFragmentChooser
[13:02:56] Running LargestFragmentChooser
[13:03:01] Running LargestFragmentChooser
[13:03:04] Running LargestFragmentChooser


...

[13:10:27] Running LargestFragmentChooser
[13:10:32] Running LargestFragmentChooser
[13:10:38] Running LargestFragmentChooser
[13:10:42] Running LargestFragmentChooser
[13:10:47] Running LargestFragmentChooser
[13:11:02] Running LargestFragmentChooser


[CHECK] molecule5.png                1/6 agree  OCC1O[C@@H](O)C(O)[C@@H](O)[C@@H]1O


[13:11:06] Running LargestFragmentChooser
[13:11:10] Running LargestFragmentChooser
[13:11:15] Running LargestFragmentChooser
[13:11:18] Running LargestFragmentChooser
[13:11:22] Running LargestFragmentChooser
[13:11:26] Running LargestFragmentChooser


[OK   ] molecule_clean.png           6/6 agree  Cn1c(=O)c2c(ncn2C)n(C)c1=O

================ SUMMARY ================
images processed : 11
trusted          : 9/11  (others are produced too, just flagged needs_review)
EXACT-MATCH ACC  : fill GROUND_TRUTH in cell 2 to measure real accuracy

Results CSV: /content/output/results.csv


,image,smiles,votes,agreement,confidence,in_pubchem,trust,needs_review,highres,correct
0,blurry1_aspirin.png,CC(C)C[C@@H]1CCCCC1C(C)C,3/6,0.50,0.909,None,True,False,/content/output/blurry1_aspirin_highres.png,None
1,mix1_aspirin_lowres.png,CC(=O)Oc1ccccc1C(=O)O,5/6,0.83,0.955,None,True,False,/content/output/mix1_aspirin_lowres_highres.png,None
2,mix2_paracetamol_blur.png,CC(=O)Nc1ccc(O)cc1,4/6,0.67,0.967,None,True,False,/content/output/mix2_paracetamol_blur_highres.png,None
3,mix3_ibuprofen_paper.png,CC(C)Cc1ccc(C(C)C(=O)O)cc1,6/6,1.00,0.992,None,True,False,/content/output/mix3_ibuprofen_paper_highres.png,None
4,mix4_caffeine_scan.png,Cn1c(=O)c2c(ncn2C)n(C)c1=O,6/6,1.00,0.995,None,True,False,/content/output/mix4_caffeine_scan_highres.png,None
5,molecule.png,Cn1c(=O)c2c(ncn2C)n(C)c1=O,6/6,1.00,0.996,None,True,False,/content/output/molecule_highres.png,None
6,molecule1.png,Cn1c(=O)c2c(nc(CC34CC5CC(CC(C5)C3)C4)n2C)n(C)c1=O,1/3,0.33,0.821,None,False,True,/content/output/molecule1_highres.png,None
7,molecule2.png,c1ccccc1,4/6,0.67,0.856,None,True,False,/content/output/molecule2_highres.png,None
8,molecule4.png,CC(=O)OCc1cccc(I)c1C(=O)O,4/6,0.67,0.951,None,True,True,/content/output/molecule4_highres.png,None
9,molecule5.png,OCC1O[C@@H](O)C(O)[C@@H](O)[C@@H]1O,1/6,0.17,0.881,None,False,True,/content/output/molecule5_highres.png,None
